# 23. HR 퇴사 예측 배깅 앙상블 (Bagging HR)

`hr_data.csv`의 퇴사 여부(`attrition`)를 타겟으로 분리(누수 방지)한 뒤 KNN, DecisionTree, SVM 기반 `BaggingClassifier`의 분류 성능을 비교합니다.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

candidates = [
    Path('Study_Python/[7] Machine_Learning/data/hr_data.csv'),
    Path('../[7] Machine_Learning/data/hr_data.csv'),
    Path('data/hr_data.csv'),
]
csv_path = next((p for p in candidates if p.is_file()), None)
data = pd.read_csv(csv_path)
y = data['attrition'].map({'No': 0, 'Yes': 1})
numeric_features = [c for c in data.select_dtypes(include=['int64', 'float64']).columns if c != 'attrition']
X = data[numeric_features].dropna(axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## 문제 1. KNN 및 Decision Tree 기반 배깅 분류기 비교

타겟이 제거된 `X_train, y_train`으로 KNN(`StandardScaler` 파이프라인) 기반 `BaggingClassifier`와 `DecisionTreeClassifier` 기반 `BaggingClassifier`를 학습해 테스트 정확도를 출력하세요.

In [ ]:
knn_bag = BaggingClassifier(
    estimator=Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))]),
    n_estimators=10, random_state=42
)
dt_bag = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=10, random_state=42)
knn_bag.fit(X_train, y_train)
dt_bag.fit(X_train, y_train)
print(f'Bagging KNN Accuracy: {accuracy_score(y_test, knn_bag.predict(X_test)):.4f}')
print(f'Bagging DT Accuracy:  {accuracy_score(y_test, dt_bag.predict(X_test)):.4f}')

**결과 해석**

타겟 열(`attrition`)을 입력 특성에서 제외하고 거리 기반 모델에 파이프라인 표준화를 적용해 누수 없는 공정한 배깅 성능을 비교할 수 있습니다.

## 문제 2. SVM 기반 배깅 분류기 학습 및 평가

`Pipeline([StandardScaler, SVC(probability=True, random_state=42)])`를 기반 모델로 하는 `BaggingClassifier(n_estimators=10, random_state=42)`의 테스트 정확도를 출력하세요.

In [ ]:
svm_bag = BaggingClassifier(
    estimator=Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
    n_estimators=10, random_state=42
)
svm_bag.fit(X_train, y_train)
print(f'Bagging SVM Accuracy: {accuracy_score(y_test, svm_bag.predict(X_test)):.4f}')

**결과 해석**

부트스트랩 표본마다 서로 다른 결정 경계를 학습한 SVM들을 배깅으로 결합하면 단일 SVM 대비 분산을 완화해 안정적인 예측을 수행합니다.